# Lesson 05 — Visualizing and Evaluating Matches

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img1 = cv2.imread('sample.jpg')
img2 = img1.copy()
M = cv2.getRotationMatrix2D((img1.shape[1]//2,img1.shape[0]//2),20,0.85)
img2 = cv2.warpAffine(img2, M, (img2.shape[1], img2.shape[0]))

sift = cv2.SIFT_create(nfeatures=300)
kp1,d1 = sift.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), None)
kp2,d2 = sift.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), None)
bf     = cv2.BFMatcher()
good   = [m for m,n in bf.knnMatch(d1,d2,k=2) if m.distance < 0.75*n.distance]

# Different drawing modes
top20 = sorted(good, key=lambda x:x.distance)[:20]
vis1 = cv2.drawMatches(img1,kp1,img2,kp2,top20,None,
                        matchColor=(0,255,0), singlePointColor=(0,0,255), flags=0)
vis2 = cv2.drawMatches(img1,kp1,img2,kp2,top20,None,flags=2)  # no single points

# Distance histogram
distances = [m.distance for m in good]
plt.figure(figsize=(8,4))
plt.hist(distances, bins=30, color='steelblue', edgecolor='none')
plt.xlabel('Match distance'); plt.ylabel('Count')
plt.title('Distribution of match distances — left = confident matches')
plt.show()

fig, axes = plt.subplots(1,2,figsize=(18,5))
axes[0].imshow(cv2.cvtColor(vis1, cv2.COLOR_BGR2RGB)); axes[0].set_title('With single points shown'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis2, cv2.COLOR_BGR2RGB)); axes[1].set_title('Matches only'); axes[1].axis('off')
plt.show()
print(f"Good matches: {len(good)}  |  Avg distance: {np.mean(distances):.1f}")

## Key Takeaway
Always visualize matches before using them. A good match set should show consistent geometric transformation (all lines roughly parallel). Random chaotic lines = bad matches.